# S6E2 TabNet
TabNet uses sequential attention to select features at each step — architecturally
different from GBTs, giving genuine diversity for ensembling.

**Strategy**: 5-fold × 3 repeats = 15 models. Average all test predictions
to reduce neural net variance. OOF predictions feed into `s6e2-ensemble.ipynb`.

**Baseline to beat**: catboost_default_proba LB = 0.95356

## Imports & Data

In [1]:
import numpy as np
import pandas as pd
import subprocess
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  "
      f"|  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

KAGGLE_DATA = Path("/kaggle/input/playground-series-s6e2")
LOCAL_DATA  = Path("data")
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    if "heart_disease" in df.columns:
        df["heart_disease"] = df["heart_disease"].map({"Absence": 0, "Presence": 1})
    return df

train = prep(pd.read_csv(DATA_DIR / "train.csv"))
test  = prep(pd.read_csv(DATA_DIR / "test.csv"))
ss    = pd.read_csv(DATA_DIR / "sample_submission.csv")

FEATURES     = [c for c in train.columns if c not in ["heart_disease", "id"]]
CAT_FEATURES = ["sex", "chest_pain_type", "fbs_over_120", "ekg_results",
                "exercise_angina", "slope_of_st", "number_of_vessels_fluro", "thallium"]
CAT_IDXS     = [FEATURES.index(c) for c in CAT_FEATURES]
CAT_DIMS     = []  # filled after encoding

X_raw = train[FEATURES].values
y     = train["heart_disease"].values
X_test_raw = test[FEATURES].values
print(f"Train: {X_raw.shape}  Test: {X_test_raw.shape}")

PyTorch 2.0.1+cu118  |  CUDA: True  |  GPU: NVIDIA GeForce RTX 3060
Train: (630000, 13)  Test: (270000, 13)


## Encode Categoricals
TabNet needs integer-encoded categoricals (0-indexed) plus the number of unique
values per categorical feature for its embedding layers.

In [2]:
# Ordinal-encode categoricals; leave numerics as float
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

X_cats      = enc.fit_transform(train[CAT_FEATURES])
X_test_cats = enc.transform(test[CAT_FEATURES])

# CAT_DIMS: number of unique values per cat feature (TabNet embeds each)
CAT_DIMS = [int(len(cats)) + 1 for cats in enc.categories_]  # +1 for unknown

# Reconstruct full feature arrays with encoded cats
NUM_FEATURES = [f for f in FEATURES if f not in CAT_FEATURES]
X_nums      = train[NUM_FEATURES].values.astype(np.float32)
X_test_nums = test[NUM_FEATURES].values.astype(np.float32)

# Reorder: put categoricals first (matching CAT_IDXS), then numerics
# TabNet wants cat features at the indices specified by cat_idxs
X      = train[FEATURES].copy()
X_test = test[FEATURES].copy()
for i, col in enumerate(CAT_FEATURES):
    X[col]      = X_cats[:, i]
    X_test[col] = X_test_cats[:, i]

X      = X.values.astype(np.float32)
X_test = X_test.values.astype(np.float32)

print(f"cat_idxs: {CAT_IDXS}")
print(f"cat_dims: {CAT_DIMS}")

cat_idxs: [1, 2, 5, 6, 8, 10, 11, 12]
cat_dims: [3, 5, 3, 4, 3, 4, 5, 4]


## TabNet Hyperparameters
- `n_d` / `n_a`: width of decision/attention embedding (keep equal, 32–64)
- `n_steps`: sequential attention steps — more = more complex (3–6)
- `gamma`: feature reuse coefficient (1.0–2.0; higher = more reuse)
- `cat_emb_dim`: embedding size per categorical value

In [3]:
TABNET_PARAMS = dict(
    n_d=32, n_a=32,
    n_steps=4,
    gamma=1.3,
    cat_idxs=CAT_IDXS,
    cat_dims=CAT_DIMS,
    cat_emb_dim=4,
    n_independent=2,
    n_shared=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_params=dict(T_max=100, eta_min=1e-4),
    mask_type="sparsemax",
    device_name="cuda" if torch.cuda.is_available() else "cpu",
    verbose=0,
    seed=42,
)
FIT_PARAMS = dict(
    max_epochs=200,
    patience=20,           # early stopping
    batch_size=4096,
    virtual_batch_size=512,
    eval_metric=["auc"],
)
print("TabNet params ready")

TabNet params ready


## 5-Fold × 3 Repeats = 15 Models
Higher repeat count reduces neural net variance in test predictions.
OOF predictions are saved for ensemble use.

In [4]:
N_REPEATS = 1
N_FOLDS   = 5

oof_tabnet  = np.zeros(len(X))
oof_counts  = np.zeros(len(X))  # track how many times each row is in val
test_preds  = np.zeros(len(X_test))
fold_aucs   = []

for repeat in range(N_REPEATS):
    kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42 + repeat)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr,  X_val  = X[tr_idx],  X[val_idx]
        y_tr,  y_val  = y[tr_idx],  y[val_idx]

        params = {**TABNET_PARAMS, "seed": 42 + repeat * 10 + fold}
        m = TabNetClassifier(**params)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            **FIT_PARAMS
        )

        val_proba  = m.predict_proba(X_val)[:, 1]
        test_proba = m.predict_proba(X_test)[:, 1]

        oof_tabnet[val_idx] += val_proba
        oof_counts[val_idx] += 1
        test_preds           += test_proba

        auc = roc_auc_score(y_val, val_proba)
        fold_aucs.append(auc)
        print(f"repeat {repeat+1} fold {fold+1}: "
              f"best_epoch={m.best_epoch}  auc={auc:.4f}")

# Average across repeats
oof_tabnet /= oof_counts
test_preds /= (N_REPEATS * N_FOLDS)

oof_auc = roc_auc_score(y, oof_tabnet)
print(f"\nTabNet OOF AUC ({N_REPEATS}×{N_FOLDS} repeats): {oof_auc:.5f}")
print(f"Mean fold AUC: {np.mean(fold_aucs):.5f}  Std: {np.std(fold_aucs):.5f}")
print(f"vs CatBoost OOF: ~0.95540")

Stop training because you reached max_epochs = 100 with best_epoch = 95 and best_val_0_auc = 0.95411


/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


repeat 1 fold 1: best_epoch=95  auc=0.9541
Stop training because you reached max_epochs = 100 with best_epoch = 97 and best_val_0_auc = 0.95305


/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


repeat 1 fold 2: best_epoch=97  auc=0.9531
Stop training because you reached max_epochs = 100 with best_epoch = 98 and best_val_0_auc = 0.95398


...

repeat 1 fold 3: best_epoch=98  auc=0.9540
Stop training because you reached max_epochs = 100 with best_epoch = 96 and best_val_0_auc = 0.95329


/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


repeat 1 fold 4: best_epoch=96  auc=0.9533
Stop training because you reached max_epochs = 100 with best_epoch = 98 and best_val_0_auc = 0.95401


/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


repeat 1 fold 5: best_epoch=98  auc=0.9540

TabNet OOF AUC (1×5 repeats): 0.95369
Mean fold AUC: 0.95369  Std: 0.00043
vs CatBoost OOF: ~0.95540


## Save OOF & Test Predictions for Ensemble

In [5]:
np.save("submissions/oof_tabnet.npy",  oof_tabnet)
np.save("submissions/test_tabnet.npy", test_preds)
print("Saved OOF and test predictions")
print(f"oof_tabnet shape:  {oof_tabnet.shape}")
print(f"test_preds shape:  {test_preds.shape}")

Saved OOF and test predictions
oof_tabnet shape:  (630000,)
test_preds shape:  (270000,)


## Submit TabNet Standalone

In [6]:
sub = ss.copy()
sub["Heart Disease"] = test_preds
fname = "submissions/tabnet_5x3.csv"
desc  = f"tabnet_5x3 | cv_auc={oof_auc:.4f}"
sub.to_csv(fname, index=False)

r = subprocess.run(
    ["kaggle", "competitions", "submit", "-c", "playground-series-s6e2",
     "-f", fname, "-m", desc],
    capture_output=True, text=True
)
status = "submitted" if "successfully" in r.stdout.lower() else r.stdout.strip()[:80]
print(f"tabnet_5x3: {status}")
print(f"  desc: {desc}")

tabnet_5x3: submitted
  desc: tabnet_5x3 | cv_auc=0.9537


## Quick GBT + TabNet Ensemble
If TabNet OOF AUC is ≥ 0.950, it's worth ensembling with GBTs.
Load saved GBT OOF predictions from the ensemble notebook if available,
otherwise use simple average of standalone test predictions.

In [8]:
from pathlib import Path
from scipy.optimize import minimize

# Load GBT OOF predictions if available from ensemble notebook
oof_files = {
    "catboost": "submissions/oof_cat.npy",
    "lgbm":     "submissions/oof_lgb.npy",
    "xgb":      "submissions/oof_xgb.npy",
}
test_files = {
    "catboost": "submissions/test_cat.npy",
    "lgbm":     "submissions/test_lgb.npy",
    "xgb":      "submissions/test_xgb.npy",
}

available = {k: Path(v).exists() for k, v in oof_files.items()}
print("GBT OOF files available:", available)

if all(available.values()):
    oofs  = {k: np.load(v) for k, v in oof_files.items()}
    tests = {k: np.load(v) for k, v in test_files.items()}
    oofs["tabnet"]  = oof_tabnet
    tests["tabnet"] = test_preds

    names  = list(oofs.keys())
    oof_list  = list(oofs.values())
    test_list = list(tests.values())

    def neg_auc(w, oofs):
        return -roc_auc_score(y, sum(wi*o for wi, o in zip(w, oofs)))

    res = minimize(
        neg_auc, x0=[0.25]*4, args=(oof_list,),
        method="SLSQP",
        bounds=[(0,1)]*4,
        constraints={"type":"eq", "fun": lambda w: w.sum()-1}
    )
    opt_weights = res.x
    opt_auc     = -res.fun
    print("\nSLSQP weights (GBTs + TabNet):")
    for n, w in zip(names, opt_weights):
        print(f"  {n:<12} {w:.4f}")
    print(f"  OOF AUC = {opt_auc:.5f}")
    print(f"  vs GBT-only ensemble: ~0.95537")

    if opt_auc > 0.95540:  # only submit if genuinely better
        pred = sum(w*t for w, t in zip(opt_weights, test_list))
        sub2 = ss.copy()
        sub2["Heart Disease"] = pred
        fname2 = "submissions/ensemble_gbt_tabnet.csv"
        desc2  = f"ensemble_gbt_tabnet | cv_auc={opt_auc:.4f}"
        sub2.to_csv(fname2, index=False)
        r2 = subprocess.run(
            ["kaggle", "competitions", "submit", "-c", "playground-series-s6e2",
             "-f", fname2, "-m", desc2],
            capture_output=True, text=True
        )
        status2 = "submitted" if "successfully" in r2.stdout.lower() else r2.stdout.strip()[:80]
        print(f"\nensemble_gbt_tabnet: {status2}")
    else:
        print(f"\nTabNet didn't improve ensemble — skipping submission")
else:
    print("Run s6e2-ensemble.ipynb first to generate GBT OOF files, then rerun this cell")

GBT OOF files available: {'catboost': True, 'lgbm': True, 'xgb': True}

SLSQP weights (GBTs + TabNet):
  catboost     0.2500
  lgbm         0.2500
  xgb          0.2500
  tabnet       0.2500
  OOF AUC = 0.95521
  vs GBT-only ensemble: ~0.95537

TabNet didn't improve ensemble — skipping submission
